In [ ]:
!pip install -q sentence-transformers

In [ ]:
from pathlib import Path
import os
import re
import json
import time
import joblib
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

from pathlib import Path
from sentence_transformers import SentenceTransformer

import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, multilabel_confusion_matrix, average_precision_score, label_ranking_average_precision_score

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

from google import genai
from google.genai import types
import os
import getpass


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [ ]:
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


In [ ]:

%pwd

'/content/drive/MyDrive/EMP/Notebook'

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [1]:
df = pd.read_csv(CSV_PATH + "Generated_Sentences.csv")
print(df.shape)
df.head()

NameError: name 'pd' is not defined

In [ ]:
df = df.drop(columns = ['ID'])

In [ ]:
df.columns.tolist()

['Campus', 'Comment']

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\r", " ").replace("\n"," ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_sentences(text):
    text = clean_text(text)
    if not text:
        return []

    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

sentence_rows = []

for source_id, row in df.reset_index(drop=True).iterrows():
    campus = row.get("Campus", "")
    comment = row.get("Comment", "")
    sentences = split_into_sentences(comment)

    for sentence_index, sentence in enumerate(sentences):
        sentence_rows.append({
            "source_id": source_id,
            "campus": campus,
            "sentence_index": sentence_index,
            "sentence_id": f"{source_id}_{sentence_index}",
            "sentence": sentence
        })
sent_df = pd.DataFrame(sentence_rows)

print(sent_df.shape)
sent_df.head(10)

(154, 5)


,source_id,campus,sentence_index,sentence_id,sentence
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...
1,1,Chico,1,1_1,I think it is important to know how each indiv...
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...
3,1,Chico,3,1_3,Since we all have so many different identities...
4,1,Chico,4,1_4,I want to continue listening to my students in...
5,2,Chico,0,2_0,"As an intersectional scholar, I had experience..."
6,2,Chico,1,2_1,"However, one highlight of this exercise for me..."
7,2,Chico,2,2_2,To be able to list the various identities with...
8,2,Chico,3,2_3,An insight I gained from this activity has to ...
9,2,Chico,4,2_4,I identify as agnostic and never really consid...


In [ ]:
sent_df.to_csv(CSV_PATH + "Module 1 sentence.csv", index = False)
print(f"saved sentence-level data to: {CSV_PATH}")

saved sentence-level data to: ../Data/


In [ ]:
import getpass
import os
from google import genai

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for model in client.models.list():
    print(model.name)
    print("supported actions:", getattr(model, "supported_actions", None))
    print("-" * 80)

Enter Gemini API key: ··········
models/gemini-2.5-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.5-pro
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-lite-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
-------

In [ ]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [ ]:
LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

LABEL_DEFINITIONS = """
You are labeling sentences for a qualitative research project.

Classify how much the sentence leans toward each of four forms of oppression.

1. Ideological oppression:
Belief systems, dominant ideas, stereotypes, norms, assumptions, cultural narratives,
or values that justify inequality.

2. Institutionalized oppression:
Oppression embedded in systems, institutions, policies, schools, religion, workplaces,
government, laws, programs, access to resources, or formal structures.

3. Interpersonal oppression:
Oppression expressed through person-to-person interaction, exclusion, judgment,
bias, conflict, mistreatment, microaggressions, or treatment by others.

4. Internalized oppression:
Oppression absorbed into the self, including shame, self-doubt, hiding identity,
discomfort with identity, accepting negative beliefs about oneself, or seeing oneself
as lesser due to dominant narratives.

For each oppression type, return:
- a binary value: 1 if the sentence clearly leans toward that oppression type, otherwise 0
- a score from 0.0 to 1.0 showing how strongly the sentence leans toward that type

Important:
- Do not force a label.
- If the sentence does not clearly lean toward any of the four oppression types, all binary labels should be 0.
- Activity feedback alone should usually receive all 0s.
- General identity reflection without oppression should usually receive all 0s.
- Teaching application should only receive a label if it clearly mentions systems, bias, exclusion, inequity, or oppression.
- Do not create a fifth category.
"""

In [ ]:
def build_prompt(sentence):
    return f"""
{LABEL_DEFINITIONS}

Return only valid JSON in this exact format:

{{
  "ideological": 0,
  "institutionalized": 0,
  "interpersonal": 0,
  "internalized": 0,
  "ideological_score": 0.0,
  "institutionalized_score": 0.0,
  "interpersonal_score": 0.0,
  "internalized_score": 0.0,
  "primary_leaning": "none",
  "rationale": "brief explanation"
}}

Sentence:
\"\"\"{sentence}\"\"\"
"""

def safe_json_parse(text):
    """
    Tries to parse Gemini output as JSON
    Also handles cases where the model wraps JSON in markdown.
    """

    if text is None:
        return None

    text = text.strip()

    # Remove markdown fences if present
    text = re.sub(f"^```json","",text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$","", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting the first JSON object
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None

def label_sentence_with_gemini(sentence, max_retries = 3, sleep_seconds = 2):
    prompt = build_prompt(sentence)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model = GEMINI_MODEL,
                contents = prompt,
                config=types.GenerateContentConfig(
                    temperature  = 0,
                    response_mime_type="application/json"
                )
            )

            parsed = safe_json_parse(response.text)

            if parsed is None:
                raise ValueError(f"Could not parse JSON: {response.text}")

            # normalize labels
            result = {}
            for label in LABELS:
                # Binary 0/1 label
                result[label] = int(parsed.get(label, 0))

                # Leaning score
                score_col = f"{label}_score"
                result[score_col] = float(parsed.get(score_col, 0.0))

            # If no label is selected, mark none or unclear
            if sum(result[label] for label in LABELS) == 0:
                result["primary_leaning"] = "none"

            else:
                result["primary_leaning"] = max(
                    LABELS,
                    key=lambda label: result[f"{label}_score"]
                )

            result["rationale"] = parsed.get("rationale", "")
            result["gemini_raw"] = response.text

            return result

        except Exception as e:
            if attempt == max_retries - 1:
                result = {}

                for label in LABELS:
                    result[label] = 0
                    result[f"{label}_score"] = 0.0

                result["primary_leaning"] = "none"
                result["rationale"]  = f"ERROR: {str(e)}"
                result["gemini_raw"] = ""

                return result;

            time.sleep(sleep_seconds * (attempt + 1))

In [ ]:
sample_df = sent_df.copy()

In [ ]:
labeled_rows = []

for _,row in tqdm(sample_df.iterrows(), total = len(sample_df)):
    sentence = row["sentence"]
    labels = label_sentence_with_gemini(sentence)

    combined = row.to_dict()
    combined.update(labels)

    # manual review fields
    combined["reviewed"] = 0
    combined["review_notes"] = ""

    labeled_rows.append(combined)

gemini_df = pd.DataFrame(labeled_rows)

gemini_df

  0%|          | 0/154 [00:00<?, ?it/s]

,source_id,campus,sentence_index,sentence_id,sentence,ideological,ideological_score,institutionalized,institutionalized_score,interpersonal,interpersonal_score,internalized,internalized_score,primary_leaning,rationale,gemini_raw,reviewed,review_notes
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0.0,0,0.0,0,0.0,1,0.7,internalized,The sentence reflects on a shift in self-perce...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
1,1,Chico,1,1_1,I think it is important to know how each indiv...,0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence emphasizes the importance of indi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence describes differing life experien...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
3,1,Chico,3,1_3,Since we all have so many different identities...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence discusses the complexity of inter...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,0.0,1,0.7,0,0.0,0,0.0,institutionalized,The sentence discusses interpreting generalize...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,22,San Francisco,0,22_0,"Age: 56 years, my son calls me old a number of...",0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence describes a person being called '...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
150,23,San Francisco,0,23_0,"Status: Lecturer Faculty, Instructor",0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a job title and doe...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
151,24,San Francisco,0,24_0,Gender: Male,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence is a simple statement of gender a...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
152,25,San Francisco,0,25_0,Disability: Wearing eye glasses,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a fact about disabi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,


In [ ]:
gemini_df.to_csv(CSV_PATH + "Module 1 Sentences Gemini.csv", index = False)
print(f"Saved Gemini weka labels to: {CSV_PATH}")

Saved Gemini weka labels to: ../Data/


In [ ]:
reviewed_df = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")

print(reviewed_df.shape)
reviewed_df.head()

(154, 18)


,source_id,campus,sentence_index,sentence_id,sentence,ideological,ideological_score,institutionalized,institutionalized_score,interpersonal,interpersonal_score,internalized,internalized_score,primary_leaning,rationale,gemini_raw,reviewed,review_notes
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0.0,0,0.0,0,0.0,1,0.7,internalized,The sentence reflects on a shift in self-perce...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
1,1,Chico,1,1_1,I think it is important to know how each indiv...,0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence emphasizes the importance of indi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence describes differing life experien...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
3,1,Chico,3,1_3,Since we all have so many different identities...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence discusses the complexity of inter...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,0.0,1,0.7,0,0.0,0,0.0,institutionalized,The sentence discusses interpreting generalize...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN


In [ ]:
reviewed_df = gemini_df.copy()

In [ ]:
TARGET_LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

df_model = reviewed_df.copy()

df_model.columns = df_model.columns.str.strip()

# validate required columns
required_cols = ["source_id", "sentence"] + TARGET_LABELS
missing_cols = [col for col in required_cols if col not in df_model.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {', '.join(missing_cols)}")

#Clean text

df_model["sentence"] = df_model["sentence"].fillna("").astype(str).str.strip()
df_model = df_model[df_model["sentence"] != ""].copy()

# clean labels
for label in TARGET_LABELS:
  df_model[label] = pd.to_numeric(df_model[label], errors = "coerce").fillna(0)
  df_model[label] = df_model[label].clip(0,1).astype(int)

# Use source_id as group
df_model["source_id"] = df_model["source_id"].astype(str)

#stage 1 target
df_model["label_count"] = df_model[TARGET_LABELS].sum(axis=1)
df_model["oppression_related"] = (df_model["label_count"]>0).astype(int)

print("Total rows:", len(df_model))
print("\nOppression-related distribution:")
print(df_model["oppression_related"].value_counts())

print("\nFour-label counts:")
print(df_model[TARGET_LABELS].sum())

Total rows: 154

Oppression-related distribution:
oppression_related
0    94
1    60
Name: count, dtype: int64

Four-label counts:
ideological          22
institutionalized    13
interpersonal        27
internalized         21
dtype: int64


In [ ]:
EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"

embedder =  SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device = device
)

sentences = df_model["sentence"].astype(str).tolist()

X_emb = embedder.encode(
    sentences,
    batch_size=32,
    normalize_embeddings=True,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embedding shape:", X_emb.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding shape: (154, 1024)


In [ ]:
X = X_emb
groups = df_model["source_id"].values
y_gate = df_model["oppression_related"].values
Y = df_model[TARGET_LABELS].values

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y_gate, groups=groups)
)

train_df = df_model.iloc[train_idx].copy()
test_df = df_model.iloc[test_idx].copy()

X_train = X[train_idx]
X_test = X[test_idx]

y_gate_train = train_df["oppression_related"].values
y_gate_test = test_df["oppression_related"].values

Y_train = train_df[TARGET_LABELS].values
Y_test = test_df[TARGET_LABELS].values

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("\nTrain gate distribution:")
print(train_df["oppression_related"].value_counts())

print("\nTest gate distribution:")
print(test_df["oppression_related"].value_counts())

print("\nTrain label counts:")
print(train_df[TARGET_LABELS].sum())

print("\nTest label counts:")
print(test_df[TARGET_LABELS].sum())

Train rows: 118
Test rows: 36

Train gate distribution:
oppression_related
0    72
1    46
Name: count, dtype: int64

Test gate distribution:
oppression_related
0    22
1    14
Name: count, dtype: int64

Train label counts:
ideological          16
institutionalized     9
interpersonal        24
internalized         18
dtype: int64

Test label counts:
ideological          6
institutionalized    4
interpersonal        3
internalized         3
dtype: int64


In [ ]:
gate_model = LogisticRegression(
    max_iter = 2000,
    class_weight = "balanced",
    solver = "liblinear",
    random_state = 42
)

gate_model.fit(
    X_train,
    y_gate_train,
)

print("stage 1 gate model trained.")

stage 1 gate model trained.


In [ ]:
gate_probs = gate_model.predict_proba(X_test)[:, 1]

for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    gate_preds = (gate_probs >= threshold).astype(int)

    print("=" * 80)
    print("Gate threshold:", threshold)

    print(classification_report(
        y_gate_test,
        gate_preds,
        target_names=["no_strong_leaning", "oppression_related"],
        zero_division=0
    ))

print("Average precision:", average_precision_score(y_gate_test, gate_probs))

Gate threshold: 0.2
                    precision    recall  f1-score   support

 no_strong_leaning       0.00      0.00      0.00        22
oppression_related       0.39      1.00      0.56        14

          accuracy                           0.39        36
         macro avg       0.19      0.50      0.28        36
      weighted avg       0.15      0.39      0.22        36

Gate threshold: 0.25
                    precision    recall  f1-score   support

 no_strong_leaning       0.00      0.00      0.00        22
oppression_related       0.39      1.00      0.56        14

          accuracy                           0.39        36
         macro avg       0.19      0.50      0.28        36
      weighted avg       0.15      0.39      0.22        36

Gate threshold: 0.3
                    precision    recall  f1-score   support

 no_strong_leaning       0.00      0.00      0.00        22
oppression_related       0.39      1.00      0.56        14

          accuracy             

In [ ]:
GATE_THRESHOLD = 0.60

In [ ]:
# ------------------------------------------------------------
# Stage 2: Primary-label multiclass classifier
# ------------------------------------------------------------

positive_train_mask = train_df["oppression_related"].values == 1

X_type_train = X_train[positive_train_mask]
stage2_train_df = train_df.loc[positive_train_mask].copy()

print("Stage 2 training rows:", len(stage2_train_df))

print("Stage 2 label counts:")
print(pd.Series(Y_type_train.sum(axis=0), index=TARGET_LABELS))

Stage 2 training rows: 46
Stage 2 label counts:
ideological          16
institutionalized     9
interpersonal        24
internalized         18
dtype: int64


In [ ]:
# ------------------------------------------------------------
# Build primary Stage 2 target
# ------------------------------------------------------------

SCORE_COLS = [f"{label}_score" for label in TARGET_LABELS]

def get_primary_label(row):
    primary = str(row.get("primary_leaning", "")).strip().lower()

    if primary in TARGET_LABELS:
        return primary

    # Fallback: choose the label with the highest Gemini score
    available_score_cols = [col for col in SCORE_COLS if col in row.index]

    if available_score_cols:
        scores = row[available_score_cols].astype(float).values
        best_idx = int(np.argmax(scores))
        return TARGET_LABELS[best_idx]

    # Final fallback: choose among binary labels if score columns are unavailable
    binary_values = row[TARGET_LABELS].astype(int).values
    if binary_values.sum() > 0:
        best_idx = int(np.argmax(binary_values))
        return TARGET_LABELS[best_idx]

    return "none"


stage2_train_df["primary_label"] = stage2_train_df.apply(get_primary_label, axis=1)

# Keep only valid four-label rows
stage2_train_df = stage2_train_df[
    stage2_train_df["primary_label"].isin(TARGET_LABELS)
].copy()

X_type_train = X_train[stage2_train_df.index.map(lambda idx: train_df.index.get_loc(idx))]
y_type_train = stage2_train_df["primary_label"].values

print("Stage 2 primary-label counts:")
print(pd.Series(y_type_train).value_counts())

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score

type_model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

type_model.fit(X_type_train, y_type_train)

print("Stage 2 primary-label multiclass model trained.")
print("Model classes:", type_model.classes_)

Stage 2 four-label model trained.


In [ ]:
# positive_test_mask = test_df["oppression_related"].values == 1

# X_type_test = X_test[positive_test_mask]

# Y_type_test = test_df.loc[
#     positive_test_mask,
#     TARGET_LABELS
# ].values

# type_probs_positive = type_model.predict_proba(X_type_test)

# print("Positive test rows:", len(X_type_test))
# print("Positive test label counts:")
# print(pd.Series(Y_type_test.sum(axis=0), index=TARGET_LABELS))

Positive test rows: 14
Positive test label counts:
ideological          6
institutionalized    4
interpersonal        3
internalized         3
dtype: int64


In [ ]:
# for threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
#     type_preds = (type_probs_positive >= threshold).astype(int)

#     # Ensure every positive sentence gets at least one type prediction
#     for i in range(len(type_preds)):
#         if type_preds[i].sum() == 0:
#             type_preds[i, np.argmax(type_probs_positive[i])] = 1

#     print("=" * 80)
#     print("Type threshold:", threshold)

#     print(classification_report(
#         Y_type_test,
#         type_preds,
#         target_names=TARGET_LABELS,
#         zero_division=0
#     ))

Type threshold: 0.2
                   precision    recall  f1-score   support

      ideological       0.43      1.00      0.60         6
institutionalized       0.29      1.00      0.44         4
    interpersonal       0.21      1.00      0.35         3
     internalized       0.21      1.00      0.35         3

        micro avg       0.29      1.00      0.44        16
        macro avg       0.29      1.00      0.44        16
     weighted avg       0.31      1.00      0.47        16
      samples avg       0.29      1.00      0.44        16

Type threshold: 0.25
                   precision    recall  f1-score   support

      ideological       0.43      1.00      0.60         6
institutionalized       0.29      1.00      0.44         4
    interpersonal       0.21      1.00      0.35         3
     internalized       0.21      1.00      0.35         3

        micro avg       0.29      1.00      0.44        16
        macro avg       0.29      1.00      0.44        16
     weigh

In [ ]:
# ------------------------------------------------------------
# Stage 2 evaluation on positive test rows only
# ------------------------------------------------------------

positive_test_mask = test_df["oppression_related"].values == 1

stage2_test_df = test_df.loc[positive_test_mask].copy()
X_type_test = X_test[positive_test_mask]

stage2_test_df["primary_label"] = stage2_test_df.apply(get_primary_label, axis=1)

stage2_test_df = stage2_test_df[
    stage2_test_df["primary_label"].isin(TARGET_LABELS)
].copy()

# Align X_type_test after filtering
positive_test_indices = np.where(positive_test_mask)[0]
valid_positive_test_indices = [
    i for i in positive_test_indices
    if test_df.iloc[i].name in stage2_test_df.index
]

X_type_test = X_test[valid_positive_test_indices]
y_type_test = stage2_test_df["primary_label"].values

type_test_probs_positive = type_model.predict_proba(X_type_test)
type_test_pred_positive = type_model.predict(X_type_test)

print("Positive test rows:", len(stage2_test_df))
print("\nTrue primary-label counts:")
print(pd.Series(y_type_test).value_counts())

print("\nPredicted primary-label counts:")
print(pd.Series(type_test_pred_positive).value_counts())

print("\nStage 2 primary-label classification report:")
print(classification_report(
    y_type_test,
    type_test_pred_positive,
    labels=TARGET_LABELS,
    zero_division=0
))

print("Stage 2 accuracy:", round(accuracy_score(y_type_test, type_test_pred_positive), 3))

print("Stage 2 top-2 accuracy:", round(
    top_k_accuracy_score(
        y_type_test,
        type_test_probs_positive,
        k=2,
        labels=type_model.classes_
    ),
    3
))

In [ ]:
TYPE_THRESHOLD = 0.55

In [ ]:
def top_k_accuracy(Y_true, probs, k=1):
    top_k_indices = np.argsort(probs, axis=1)[:, -k:]

    hits = []

    for i, indices in enumerate(top_k_indices):
        hits.append(Y_true[i, indices].sum() > 0)

    return float(np.mean(hits))

In [ ]:
if len(Y_type_test) > 0:
    lrap = label_ranking_average_precision_score(
        Y_type_test,
        type_probs_positive
    )

    top1 = top_k_accuracy(Y_type_test, type_probs_positive, k=1)
    top2 = top_k_accuracy(Y_type_test, type_probs_positive, k=2)

    print("Stage 2 LRAP:", lrap)
    print("Stage 2 Top-1 accuracy:", top1)
    print("Stage 2 Top-2 accuracy:", top2)
else:
    print("No positive test rows available for ranking evaluation.")

Stage 2 LRAP: 0.625
Stage 2 Top-1 accuracy: 0.42857142857142855
Stage 2 Top-2 accuracy: 0.6428571428571429


In [ ]:
def combined_predict(
    gate_probs,
    type_probs,
    gate_threshold,
    type_threshold
):

  preds = np.zeros_like(gate_probs, dtype=int)

  for i in range(len(gate_probs)):
      if gate_probs[i] >= gate_threshold:
          preds[i] = (type_probs[i] >= type_threshold).astype(int)

  return preds


In [ ]:
gate_test_probs = gate_model.predict_proba(X_test)[:, 1]
type_test_probs = type_model.predict_proba(X_test)

# Combined score: "is this oppression-related?" × "which type?"
final_test_scores = type_test_probs * gate_test_probs[:, None]


def combined_predict_from_final_scores(final_scores, threshold):
    return (final_scores >= threshold).astype(int)


for final_threshold in [0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65]:
    Y_pred_combined = combined_predict_from_final_scores(
        final_scores=final_test_scores,
        threshold=final_threshold
    )

    print("=" * 80)
    print("FINAL_THRESHOLD:", final_threshold)
    print("Predicted labels:", int(Y_pred_combined.sum()))
    print("True labels:", int(Y_test.sum()))

    print(classification_report(
        Y_test,
        Y_pred_combined,
        target_names=TARGET_LABELS,
        zero_division=0
    ))

FINAL_THRESHOLD: 0.2
Predicted labels: 99
True labels: 16
                   precision    recall  f1-score   support

      ideological       0.14      0.50      0.22         6
institutionalized       0.17      1.00      0.30         4
    interpersonal       0.10      1.00      0.19         3
     internalized       0.12      1.00      0.21         3

        micro avg       0.13      0.81      0.23        16
        macro avg       0.13      0.88      0.23        16
     weighted avg       0.14      0.81      0.23        16
      samples avg       0.10      0.31      0.15        16

FINAL_THRESHOLD: 0.25
Predicted labels: 37
True labels: 16
                   precision    recall  f1-score   support

      ideological       0.11      0.17      0.13         6
institutionalized       0.12      0.25      0.17         4
    interpersonal       0.12      0.33      0.18         3
     internalized       0.25      1.00      0.40         3

        micro avg       0.16      0.38      0.23    

In [ ]:
inspect_df = test_df[
    ["source_id", "sentence_id", "sentence", "oppression_related"] + TARGET_LABELS
].copy()

inspect_df["gate_score"] = gate_test_probs
inspect_df["gate_pred"] = (gate_test_probs >= GATE_THRESHOLD).astype(int)

for i, label in enumerate(TARGET_LABELS):
    inspect_df[f"{label}_type_score"] = type_test_probs[:, i]
    inspect_df[f"{label}_final_score"] = final_test_scores[:, i]
    inspect_df[f"pred_{label}"] = Y_pred_combined[:, i]

inspect_df["predicted_primary"] = [
    TARGET_LABELS[int(np.argmax(final_test_scores[i]))]
    if gate_test_probs[i] >= GATE_THRESHOLD
    else "no_strong_leaning"
    for i in range(len(inspect_df))
]

inspect_df.sort_values("gate_score", ascending=False).head(30)